In [1]:
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

import gplately

from lib.main import *
from lib.plot import *

from parameters import parameters


##################################################################################################

            You are using a DEV version (1.3.0.post277+git.d5c56f91) GPlately.     
            Some functionalities in the DEV version have not been tested thoroughly, 
            and may break your code or produce wrong results due to 
            its unstable nature(DEV in progress). Proceed With Caution!!!
            You might also need to install the DEV version plate_model_manager 
            from https://github.com/michaelchin/plate-model-manager.

            To disable this warning, 
                set USING_DEV_VERSION to False in __init__.py 
            or
                set DISABLE_GPLATELY_DEV_WARNING environment variable to true. 
            
            For example,
                os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true" (in Python)
            or
                export DISABLE_GPLATELY_DEV_WARNING=true (in Shell)
            or 
                $env:DI

In [2]:
# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
feat_maps_dir = parameters["feat_maps_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

feat_maps_dir = os.path.join(outputs_dir, feat_maps_dir)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

nprocs = 12

In [3]:
rotation_model = [
    plate_model_dir+"/1800_1000_rotfile_20240725.rot",
    plate_model_dir+"/1000_0_rotfile_20240725.rot",
]

topology_features = [
    plate_model_dir+"/1800-1000_plate_boundaries.gpml",
    plate_model_dir+"/250-0_plate_boundaries.gpml",
    plate_model_dir+"/410-250_plate_boundaries.gpml",
    plate_model_dir+"/1000-410-Convergence.gpml",
    plate_model_dir+"/1000-410-Divergence.gpml",
    plate_model_dir+"/1000-410-plate-boundaries.gpml",
    plate_model_dir+"/1000-410-Transforms.gpml",
    plate_model_dir+"/TopologyBuildingBlocks.gpml",
]

static_polygons = plate_model_dir+"/static_polygons.gpmlz"
coastlines = plate_model_dir+"/shapes_coasts.gpmlz"
continents = plate_model_dir+"/shapes_continents.gpmlz"
COBs = plate_model_dir+"/COBfile_1800_0.gpml"

In [4]:
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        verbose=True,
    )
    
    subduction_data.to_csv(subduction_data_filename, index=False)

In [5]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove("lon")
features_plot.remove("lat")
features_plot.remove("age (Ma)")
features_plot.remove("subducting_plate_ID")
features_plot.remove("trench_plate_ID")

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
)

projection = ccrs.Mollweide(central_longitude=60)

In [6]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    ax.set_global()

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)
    
    feat = ax.scatter(subduction_data_t["lon"], subduction_data_t["lat"], 50, marker=".",
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)

    gplot.plot_all_topologies(ax, color="orangered", zorder=5)
    gplot.plot_trenches(ax, color="dimgray", zorder=6)
            
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='black', alpha=0.3, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
    
    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    cbar_feat = fig.colorbar(feat, shrink=0.4, pad=0.06, orientation="horizontal", extend="both")
    cbar_feat.set_label(format_feature_name(feature), fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor="tan", edgecolor="none", label="Continental Crust"),
        Line2D([0], [0], color="orangered", label="Mid-Ocean Ridges"),
        Line2D([0], [0], color="dimgray", label="Trench Lines")
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.2))
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

In [7]:
if os.path.exists(feat_maps_dir):
    print(f"Feature maps are located in {feat_maps_dir}")
else:
    os.makedirs(feat_maps_dir, exist_ok=True)
       
    generate_feat_maps(
        rotation_model,
        topology_features,
        static_polygons,
        coastlines,
        continents,
        COBs,
        subduction_data,
        projection,
        time_steps,
        feature="convergence_rate (cm/yr)",
        output_dir=feat_maps_dir,
        n_jobs=nprocs
    )
    
    output_filenames = [
        os.path.join(feat_maps_dir, f"feat_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]
    
    output_filename = os.path.join(outputs_dir, "feat_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

Feature maps are located in outputs\feat_maps
